# **CSCI 3202: Course Final Project - Mancala AI**

**Name:** Jeffrey Allen


**1. Abstract & Introduction**

\[TODO: Write a concise abstract (approx. 150-250 words) summarizing the
project goals, the AI algorithms implemented (Minimax and Alpha-Beta),
the heuristic strategies tested, and the primary findings regarding
computational efficiency and agent performance.\]  
**Project Overview:**  
This report documents the implementation and analysis of an artificial
intelligence agent designed to play the game of Mancala. The project
adheres to “Classic Rules,” specifically omitting the “extra turn” rule
as per the course constraints. We evaluate the effectiveness of
adversarial search techniques and quantify the performance gains
provided by Alpha-Beta pruning over standard Minimax across varying
depths.

In [1]:
import random  
import numpy as np  
import time  
import matplotlib.pyplot as plt

**Random Seed for Reproducability on Outputs**

In [2]:
random.seed(109) 

**2. Methodology: Environment Setup & Game Logic**

The following section implements the Mancala class, which serves as the
environment for our agents. It handles board state, move validation,
stone distribution, and capture logic.  

In [3]:
class Mancala:
    def __init__(self, pits_per_player=6, stones_per_pit=4):
        self.pits_per_player = pits_per_player
        self.board = [stones_per_pit] * ((pits_per_player + 1) * 2)
        self.current_player = 1
        self.moves = []
        self.p1_pits_index = [0, self.pits_per_player - 1]
        self.p1_mancala_index = self.pits_per_player
        self.p2_pits_index = [self.pits_per_player + 1, len(self.board) - 2]
        self.p2_mancala_index = len(self.board) - 1
        
        # Track the winner (-1: ongoing, 0: tie, 1: P1, 2: P2)
        self.winner = -1
        
        self.board[self.p1_mancala_index] = 0
        self.board[self.p2_mancala_index] = 0

    def display_board(self):
        player_1_pits = self.board[self.p1_pits_index[0]: self.p1_pits_index[1] + 1]
        player_1_mancala = self.board[self.p1_mancala_index]
        player_2_pits = self.board[self.p2_pits_index[0]: self.p2_pits_index[1] + 1]
        player_2_mancala = self.board[self.p2_mancala_index]

        print('P1               P2')
        print('     ____{}____     '.format(player_2_mancala))
        for i in range(self.pits_per_player):
            print('{} -> | {} | {} | <- {}'.format(i + 1, player_1_pits[i], player_2_pits[-(i + 1)], self.pits_per_player - i))
        print('         {}         '.format(player_1_mancala))
        print('Turn: P' + str(self.current_player))
        
    def valid_move(self, pit):
        if not (1 <= pit <= self.pits_per_player):
            return False
        index = (pit - 1) if self.current_player == 1 else (self.pits_per_player + pit)
        return self.board[index] > 0
        
    def random_move_generator(self):
        valid_pits = [p for p in range(1, self.pits_per_player + 1) if self.valid_move(p)]
        return random.choice(valid_pits) if valid_pits else None
    
    def play(self, pit, silent=False):
        if not self.valid_move(pit):
            return self.board
            
        index = (pit - 1) if self.current_player == 1 else (self.pits_per_player + pit)
        stones = self.board[index]
        self.board[index] = 0
        self.moves.append((self.current_player, pit))

        curr_index = index
        while stones > 0:
            curr_index = (curr_index + 1) % len(self.board)
            if (self.current_player == 1 and curr_index == self.p2_mancala_index) or \
               (self.current_player == 2 and curr_index == self.p1_mancala_index):
                continue
            self.board[curr_index] += 1
            stones -= 1

        # Capture logic
        if self.board[curr_index] == 1:
            if (self.current_player == 1 and self.p1_pits_index[0] <= curr_index <= self.p1_pits_index[1]) or \
               (self.current_player == 2 and self.p2_pits_index[0] <= curr_index <= self.p2_pits_index[1]):
                opp_index = (2 * self.pits_per_player) - curr_index
                if self.board[opp_index] > 0:
                    captured = self.board[curr_index] + self.board[opp_index]
                    self.board[curr_index] = 0
                    self.board[opp_index] = 0
                    mancala_idx = self.p1_mancala_index if self.current_player == 1 else self.p2_mancala_index
                    self.board[mancala_idx] += captured

        self.current_player = 2 if self.current_player == 1 else 1 
        return self.board
    
    def winning_eval(self, silent=False):
        p1_stones = sum(self.board[self.p1_pits_index[0]: self.p1_pits_index[1] + 1])
        p2_stones = sum(self.board[self.p2_pits_index[0]: self.p2_pits_index[1] + 1])
        
        if p1_stones == 0 or p2_stones == 0:
            self.board[self.p1_mancala_index] += p1_stones
            self.board[self.p2_mancala_index] += p2_stones
            for i in range(len(self.board)):
                if i != self.p1_mancala_index and i != self.p2_mancala_index:
                    self.board[i] = 0
            
            # Determine and set the winner
            p1_score = self.board[self.p1_mancala_index]
            p2_score = self.board[self.p2_mancala_index]
            if p1_score > p2_score:
                self.winner = 1
            elif p2_score > p1_score:
                self.winner = 2
            else:
                self.winner = 0
            return True
        return False

**3. Part 2: Baseline Performance (Random vs. Random)**

To measure the complexity of the game and check for inherent bias, we
simulate 100 matches where both players select moves randomly.  

In [4]:
num_games = 100
results = {"P1_wins": 0, "P2_wins": 0, "Ties": 0, "Total_Turns": []}

for _ in range(num_games):
    game = Mancala()
    turns = 0
    while not game.winning_eval():
        move = game.random_move_generator()
        if move is None: break
        game.play(move, silent=True)
        turns += 1
    
    results["Total_Turns"].append(turns)
    p1_score = game.board[game.p1_mancala_index]
    p2_score = game.board[game.p2_mancala_index]
    
    if p1_score > p2_score:
        results["P1_wins"] += 1
    elif p2_score > p1_score:
        results["P2_wins"] += 1
    else:
        results["Ties"] += 1

# Report Statistics
print(f"Results for {num_games} Games:")
print(f"Player 1: Wins: {results['P1_wins']} ({results['P1_wins']/num_games:.1%}), "
      f"Losses: {results['P2_wins']} ({results['P2_wins']/num_games:.1%}), "
      f"Ties: {results['Ties']} ({results['Ties']/num_games:.1%})")
print(f"Player 2: Wins: {results['P2_wins']} ({results['P2_wins']/num_games:.1%}), "
      f"Losses: {results['P1_wins']} ({results['P1_wins']/num_games:.1%}), "
      f"Ties: {results['Ties']} ({results['Ties']/num_games:.1%})")
print(f"Average turns per game: {np.mean(results['Total_Turns']):.2f}")

Results for 100 Games:
Player 1: Wins: 45 (45.0%), Losses: 51 (51.0%), Ties: 4 (4.0%)
Player 2: Wins: 51 (51.0%), Losses: 45 (45.0%), Ties: 4 (4.0%)
Average turns per game: 45.82


## Intermediate Analysis & Conclusion

**1. Statistics Summary:**
* **Player 1 Win Rate:** 45%
* **Player 2 Win Rate:** 51%
* **Average Turns:** 45.82

**2. Is there a first move advantage?**
Based on the results of 100 random vs. random games, there is not an advntage as shown by Player 1's greater losses within the game's output. 


**4. Part 3 & 4: AI Algorithm Implementation**

Implementation of standard Minimax and the optimized Alpha-Beta Pruning
algorithm. Both utilize a basic heuristic evaluating the score
differential. 

In [5]:
import random
import time

def clone_game(game):
    """Creates a fast clone of the game state for AI look-ahead."""
    new_game = Mancala(pits_per_player=game.pits_per_player, stones_per_pit=0)
    new_game.board = list(game.board)
    new_game.current_player = game.current_player
    new_game.moves = list(game.moves)
    new_game.winner = game.winner 
    return new_game

def heuristic(game, maximizing_player):
    """Utility = Stones in Max Mancala - Stones in Min Mancala."""
    p1_mancala = game.board[game.p1_mancala_index]
    p2_mancala = game.board[game.p2_mancala_index]
    return (p1_mancala - p2_mancala) if maximizing_player == 1 else (p2_mancala - p1_mancala)

def minimax_search(game, depth, maximizing_player):
    if depth == 0 or game.winning_eval(silent=True):
        return heuristic(game, maximizing_player), None
    valid_moves = [p for p in range(1, game.pits_per_player + 1) if game.valid_move(p)]
    if not valid_moves: return heuristic(game, maximizing_player), None
    
    best_move = random.choice(valid_moves)
    if game.current_player == maximizing_player:
        max_eval = -float('inf')
        for move in valid_moves:
            child = clone_game(game)
            child.play(move, silent=True)
            eval_val, _ = minimax_search(child, depth - 1, maximizing_player)
            if eval_val > max_eval:
                max_eval = eval_val
                best_move = move
        return max_eval, best_move
    else:
        min_eval = float('inf')
        for move in valid_moves:
            child = clone_game(game)
            child.play(move, silent=True)
            eval_val, _ = minimax_search(child, depth - 1, maximizing_player)
            if eval_val < min_eval:
                min_eval = eval_val
                best_move = move
        return min_eval, best_move

def alpha_beta_search(game, depth, maximizing_player, alpha=-float('inf'), beta=float('inf')):
    if depth == 0 or game.winning_eval(silent=True):
        return heuristic(game, maximizing_player), None
    valid_moves = [p for p in range(1, game.pits_per_player + 1) if game.valid_move(p)]
    if not valid_moves: return heuristic(game, maximizing_player), None

    best_move = random.choice(valid_moves)
    if game.current_player == maximizing_player:
        max_eval = -float('inf')
        for move in valid_moves:
            child = clone_game(game)
            child.play(move, silent=True)
            eval_val, _ = alpha_beta_search(child, depth - 1, maximizing_player, alpha, beta)
            if eval_val > max_eval:
                max_eval = eval_val
                best_move = move
            alpha = max(alpha, eval_val)
            if beta <= alpha: break
        return max_eval, best_move
    else:
        min_eval = float('inf')
        for move in valid_moves:
            child = clone_game(game)
            child.play(move, silent=True)
            eval_val, _ = alpha_beta_search(child, depth - 1, maximizing_player, alpha, beta)
            if eval_val < min_eval:
                min_eval = eval_val
                best_move = move
            beta = min(beta, eval_val)
            if beta <= alpha: break
        return min_eval, best_move

def simulate_games(num_games, p1_type, p2_type, p1_depth=None, p2_depth=None):
    results = {"P1_wins": 0, "P2_wins": 0, "Ties": 0, "Total_Turns": [], "Durations": []}
    for i in range(num_games):
        game = Mancala()
        start_time = time.time() # This was causing the error
        while not game.winning_eval(silent=True):
            if game.current_player == 1:
                if p1_type == 'random': move = game.random_move_generator()
                elif p1_type == 'alphabeta': _, move = alpha_beta_search(game, p1_depth, 1)
                elif p1_type == 'minimax': _, move = minimax_search(game, p1_depth, 1)
            else:
                if p2_type == 'random': move = game.random_move_generator()
                elif p2_type == 'alphabeta': _, move = alpha_beta_search(game, p2_depth, 2)
                elif p2_type == 'minimax': _, move = minimax_search(game, p2_depth, 2)
            if move is None: break
            game.play(move, silent=True)
        results["Durations"].append(time.time() - start_time)
        results["Total_Turns"].append(len(game.moves))
        if game.winner == 1: results["P1_wins"] += 1
        elif game.winner == 2: results["P2_wins"] += 1
        else: results["Ties"] += 1
        if (i + 1) % 25 == 0: print(f"Progress: {i + 1}/{num_games}")
    return results

**5. Part 5: Algorithm Efficiency Analysis**

This section compares the execution time and effectiveness of Minimax
versus Alpha-Beta at a fixed depth of 5 plies.

In [6]:
**5. Part 5: Algorithm Efficiency Analysis**

This section compares the execution time and effectiveness of Minimax
versus Alpha-Beta at a fixed depth of 5 plies.

SyntaxError: invalid syntax (2483715303.py, line 1)